# 🔧 Manual LLM Tool Orchestration (LangChain)

This notebook demonstrates a **manual implementation of a multi-step tool-calling loop** using LangChain and Azure OpenAI.

---

## 🧠 What this does

Instead of relying on built-in agents, this implementation manually handles the full tool-calling lifecycle:

1. **User Query → LLM**
2. **LLM decides which tools to call (`tool_calls`)**
3. **Tools are executed manually in Python**
4. **Results are sent back to the LLM using `ToolMessage`**
5. **LLM continues reasoning until a final answer is produced**

---

## ⚙️ Key Concepts Covered

- **Custom Tools** using `@tool` (e.g., add, multiply)
- **External Tools** (Wikipedia, Tavily Search)
- **Tool Binding** using `llm.bind_tools()`
- **Tool Execution Layer** (dynamic mapping + `.invoke()`)
- **Structured Message Flow**:
  - `HumanMessage`
  - `AIMessage` (with `tool_calls`)
  - `ToolMessage`
- **Iterative Reasoning Loop** (`while True`)
- **Error Handling** for robust execution
- **Full Conversation State Tracking** for debugging

---

## 🔁 How it works (Flow)
User → LLM → Tool Calls → Tool Execution → LLM → ... → Final Answer

This loop continues until the LLM stops requesting tools.

---

## 🚀 Why this matters

This is essentially a **manual ReAct-style agent implementation**, helping understand:

- How LLMs plan actions
- How tools are executed and chained
- How structured messaging drives reasoning
- What happens under the hood of LangChain agents

---

## 📝 Usage

Simply modify the `query` variable in the code cell to test different multi-step tasks.

---

In [83]:
# ================================
# 🔐 Setup & Imports
# ================================
import os
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import AzureChatOpenAI
from langchain_tavily import TavilySearch
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


# ================================
# 🔐 Load Environment Variables
# ================================
load_dotenv()

AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")


# ================================
# 🤖 Initialize LLM
# ================================
llm = AzureChatOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    deployment_name="gpt-4o",
    api_version="2024-02-15-preview"
)


# ================================
# 🧰 Define Custom Tools
# ================================
@tool
def add(a: int, b: int) -> int:
    """Adds two numbers."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b


# ================================
# 🌐 External Tools
# ================================
# Wikipedia
wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=500)
)

# Tavily Search
tool_search = TavilySearch(max_results=5, topic="general")


# ================================
# 🧩 Tool Registry
# ================================
tools = [add, multiply, wiki_tool, tool_search]

# Map tool names → actual functions
tool_lookup = {tool.name: tool for tool in tools}


# ================================
# 🔗 Bind Tools to LLM
# ================================
llm_with_tools = llm.bind_tools(tools)


# ================================
# 🧠 USER QUERY (EDIT THIS ONLY)
# ================================
query = "I want to add 5 and 10, multiply the result by 2, search for the latest news on AI and also get a summary of what anti-matter is."


# ================================
# 🔁 MANUAL TOOL-CALLING LOOP
# ================================
messages = [HumanMessage(content=query)]

while True:
    # Step 1: LLM decides what to do
    response = llm_with_tools.invoke(messages)

    # Step 2: If no tools needed → final answer
    if not response.tool_calls:
        print("\n✅ FINAL ANSWER:\n")
        print(response.content)
        break

    # Step 3: Execute tools requested by LLM
    tool_messages = []

    for tool_call in response.tool_calls:
        selected_tool = tool_lookup[tool_call["name"]]

        try:
            result = selected_tool.invoke(tool_call["args"])
        except Exception as e:
            result = f"Tool failed: {str(e)}"

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

    # Step 4: Feed tool results back into conversation
    messages = messages + [response] + tool_messages

messages

c:\Users\KrishnaShukla\Desktop\Gen-AI\.venv\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\KrishnaShukla\Desktop\Gen-AI\.venv\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')



✅ FINAL ANSWER:

Here's the outcome of your requests:

1. Adding **5** and **10** gives **15**, and multiplying it by **2** results in **30**.

2. A quick summary of antimatter:
   - Antimatter consists of antiparticles (partners of regular particles with reversed charges and parity). It occurs naturally in cosmic ray collisions and some types of radioactive decay. Scientists have successfully bound some of these particles together in experiments to form antiatoms, but only in minimal quantities.

3. Latest AI news highlights:
   - **NVIDIA** unveiled a flagship AI platform named "Vera Rubin," focused on massive scaling requirements for trillion-parameter models.
   - **Microsoft Research** predicts that in 2026, AI will actively collaborate in scientific discoveries, generating hypotheses and assisting in experiments.
   - **TechCrunch reported** about breakthroughs like OpenAI's GPT-5.5, advancements in AI-enabled apps, and ethical issues surrounding AI. 

Would you like to look dee

[HumanMessage(content='I want to add 5 and 10, multiply the result by 2, search for the latest news on AI and also get a summary of what anti-matter is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 1325, 'total_tokens': 1402, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1280}, 'latency_checkpoint': {'engine_tbt_ms': 16, 'engine_ttft_ms': 172, 'engine_ttlt_ms': 1427, 'pre_inference_ms': 150, 'service_tbt_ms': 16, 'service_ttft_ms': 658, 'service_ttlt_ms': 1878, 'total_duration_ms': 1736, 'user_visible_ttft_ms': 508}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DcoCFkvo6Su7jGItMJPsAgEM68aKJ', 'service_tier': 'default', 'prom